In [57]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold, StratifiedKFold, cross_val_score
from sklearn import linear_model, tree, ensemble

import os
for dirname, _, filenames in os.walk('kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

kaggle/input/train.csv
kaggle/input/test.csv


In [58]:
train_data = pd.read_csv('kaggle/input/train.csv')
# pd.set_option('display.max_columns', None)
# print(train_data.head())

print(train_data.columns.tolist())
print(train_data.info())

['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB
None


In [69]:
train_data = pd.read_csv('kaggle/input/train.csv')

train_data.dropna(axis=0, subset=['Survived'], inplace=True)
y = train_data.Survived # Target varible
train_data.drop(['Survived'], axis=1, inplace=True) # Removing target variable from training data

train_data.drop(['Age'], axis=1, inplace=True) # Remove columns with null values

# Select numeric columns only
numeric_cols = [cname for cname in train_data.columns
                if train_data[cname].dtype in ['int64', 'float64'] and cname != 'PassengerId']

# print(numeric_cols)
# print(type(numeric_cols))
# arr_1d = np.array(numeric_cols)
# print(type(arr_1d))
# print(arr_1d.shape)
# matrix = arr_1d.reshape(1, 5)
# print(matrix.shape)
# print(matrix)

X = train_data[numeric_cols].copy()

print("Shape of input data: {} and shape of target variable: {}".format(X.shape, y.shape))
pd.concat([X, y], axis=1).head() # Show first 5 training examples

Shape of input data: (891, 4) and shape of target variable: (891,)


,Pclass,SibSp,Parch,Fare,Survived
0,3,1,0,7.2500,0
1,1,1,0,71.2833,1
2,3,0,0,7.9250,1
3,1,1,0,53.1000,1
4,3,0,0,8.0500,0


In [70]:
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cnt = 1

for train_index, test_index in kf.split(X, y):
    print(f"Fold:{cnt}, Train set: {len(train_index)}, Test set: {len(test_index)}")
    cnt += 1

Fold:1, Train set: 712, Test set: 179
Fold:2, Train set: 713, Test set: 178
Fold:3, Train set: 713, Test set: 178
Fold:4, Train set: 713, Test set: 178
Fold:5, Train set: 713, Test set: 178


In [71]:
# 이 코드 한 줄 안에는 모델 학습, 예측, 그리고 채점 과정이 한 방에 압축되어 있습니다.
score = cross_val_score(tree.DecisionTreeClassifier(random_state=42),
                        X, y, cv=kf, scoring="accuracy")
print(f"Scores for each fold are: {score}")
print(f"Average score: {score.mean():.2f}")

Scores for each fold are: [0.67597765 0.71348315 0.66853933 0.65730337 0.71348315]
Average score: 0.69


In [72]:
score = cross_val_score(ensemble.RandomForestClassifier(random_state=42),
                        X, y, cv=kf, scoring="accuracy")
print(f"Scores for each fold are: {score}")
print(f"Average score: {'{:.2f}'.format(score.mean())}")

Scores for each fold are: [0.69273743 0.71348315 0.64044944 0.66292135 0.71348315]
Average score: 0.68


In [74]:
final_model = ensemble.RandomForestClassifier(random_state=42)
final_model.fit(X, y)

importances = final_model.feature_importances_

feat_importances = pd.Series(importances, index=X.columns)
print(feat_importances.sort_values(ascending=False))

Fare      0.696238
Pclass    0.137123
Parch     0.086360
SibSp     0.080279
dtype: float64


In [79]:
test_data = pd.read_csv('kaggle/input/test.csv')
# print(test_data.head())
test_data.drop(['Age'], axis=1, inplace=True)

numeric_cols = [cname for cname in test_data.columns
                if test_data[cname].dtype in ['float64', 'int64'] and cname != 'PassengerId']

x = test_data[numeric_cols].copy()
x.head()

,Pclass,SibSp,Parch,Fare
0,3,0,0,7.8292
1,3,1,0,7.0000
2,2,0,0,9.6875
3,3,0,0,8.6625
4,3,1,1,12.2875


In [80]:
final_model = ensemble.RandomForestClassifier(random_state=42)

final_model.fit(X, y)

RandomForestClassifier(random_state=42)

In [81]:
predictions = final_model.predict(x)

In [82]:
print(predictions[:10])

[0 0 0 0 1 0 0 1 0 0]


In [84]:
submission = pd.DataFrame({
    "PassengerId": test_data["PassengerId"],
    "Survived": predictions
})
submission.to_csv('submission.csv', index=False)

result = pd.read_csv("submission.csv")
result

,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,1
...,...,...
413,1305,0
414,1306,1
415,1307,0
416,1308,0
